In [0]:
from pyspark.sql.functions import col, when, count

In [0]:
!pip install Faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 49.9 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import csv
import random
from datetime import datetime, timedelta
from faker import Faker

fake = Faker()
Faker.seed(42)
random.seed(42)

N_CUSTOMERS = 550
N_PRODUCTS = 520
N_ORDERS = 900
MIN_ITEMS_PER_ORDER = 1
MAX_ITEMS_PER_ORDER = 4

OUT_DIR = "/Workspace/Users/yashikumawat53@gmail.com/Drafts/ecommerce_project/data"

CATEGORIES = {
    "Electronics": ["Mobiles", "Laptops", "Audio", "Cameras", "Accessories"],
    "Clothing": ["Men", "Women", "Kids", "Footwear", "Winterwear"],
    "Home": ["Kitchen", "Furniture", "Decor", "Bedding", "Storage"],
    "Books": ["Fiction", "Non-Fiction", "Comics", "Academic", "Children"],
}

CUSTOMER_TYPES = ["REGULAR", "PREMIUM", "VIP"]
CUSTOMER_TYPE_WEIGHTS = [0.65, 0.25, 0.10]

ORDER_STATUSES = ["PLACED", "SHIPPED", "DELIVERED", "CANCELLED", "RETURNED"]
ORDER_STATUS_WEIGHTS = [0.10, 0.15, 0.55, 0.10, 0.10]

REGION_CODES = ["NA-EAST", "NA-WEST", "EU-WEST", "EU-EAST", "APAC-IN", "APAC-SG"]


REG_START = datetime(2024, 1, 1)
REG_END = datetime(2026, 5, 1)
ORDER_START = datetime(2024, 3, 1)
ORDER_END = datetime(2026, 6, 30)


def random_date(start: datetime, end: datetime) -> datetime:
    delta = end - start
    return start + timedelta(seconds=random.randint(0, int(delta.total_seconds())))

def generate_customers():
    rows = []
    for cid in range(1, N_CUSTOMERS + 1):
        name = fake.name()
        email = f"{name.lower().replace(' ', '.')}{cid}@{fake.free_email_domain()}"

        if random.random() < 0.02:
            if random.random() < 0.5:
                email = email.replace("@", "") 
            else:
                email = email.split("@")[0] + "@" 

        reg_date = random_date(REG_START, REG_END).strftime("%Y-%m-%d")
        ctype = random.choices(CUSTOMER_TYPES, weights=CUSTOMER_TYPE_WEIGHTS)[0]

        rows.append(
            {
                "customer_id": cid,
                "customer_name": name,
                "email": email,
                "registration_date": reg_date,
                "customer_type": ctype,
            }
        )

    with open(f"{OUT_DIR}/customers.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)

    print(f"customers.csv: {len(rows)} rows written")
    return rows

def generate_products():
    rows = []
    adjectives = ["Premium", "Classic", "Deluxe", "Compact", "Pro", "Essential", "Ultra"]
    for pid in range(1, N_PRODUCTS + 1):
        category = random.choice(list(CATEGORIES.keys()))
        subcategory = random.choice(CATEGORIES[category])
        base_name = f"{random.choice(adjectives)} {subcategory[:-1] if subcategory.endswith('s') else subcategory} {fake.word().capitalize()}"

        product_name = base_name
        if random.random() < 0.15:
            messy_variant = random.random()
            if messy_variant < 0.34:
                product_name = "  " + base_name + "   "  
            elif messy_variant < 0.67:
                product_name = base_name.upper() 
            else:
                product_name = base_name.lower()

        cost_price = round(random.uniform(3, 800), 2)

        rows.append(
            {
                "product_id": pid,
                "product_name": product_name,
                "category": category,
                "subcategory": subcategory,
                "cost_price": cost_price,
            }
        )

    with open(f"{OUT_DIR}/products.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)

    print(f"products.csv: {len(rows)} rows written")
    return rows

def generate_orders(customers):
    rows = []
    customer_ids = [c["customer_id"] for c in customers]

    for oid in range(1, N_ORDERS + 1):
        if random.random() < 0.05:
            customer_id = ""  
        else:
            customer_id = random.choice(customer_ids)

        order_dt = random_date(ORDER_START, ORDER_END)

        if random.random() < 0.08:
            order_date = order_dt.strftime("%d-%m-%Y")
        else:
            order_date = order_dt.strftime("%Y-%m-%d %H:%M:%S")

        status = random.choices(ORDER_STATUSES, weights=ORDER_STATUS_WEIGHTS)[0]
        region_code = random.choice(REGION_CODES)

        rows.append(
            {
                "order_id": oid,
                "customer_id": customer_id,
                "order_date": order_date,
                "status": status,
                "region_code": region_code,
                "_order_dt": order_dt,
            }
        )

    with open(f"{OUT_DIR}/orders.csv", "w", newline="", encoding="utf-8") as f:
        fieldnames = ["order_id", "customer_id", "order_date", "status", "region_code"]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in rows:
            writer.writerow({k: r[k] for k in fieldnames})

    print(f"orders.csv: {len(rows)} rows written")
    return rows


def generate_order_items(orders, products):
    rows = []
    item_id = 1
    valid_order_ids = [o["order_id"] for o in orders]
    product_ids = [p["product_id"] for p in products]
    product_cost = {p["product_id"]: p["cost_price"] for p in products}
    max_valid_order_id = max(valid_order_ids)

    for order in orders:
        n_items = random.randint(MIN_ITEMS_PER_ORDER, MAX_ITEMS_PER_ORDER)
        chosen_products = random.sample(product_ids, k=min(n_items, len(product_ids)))

        for product_id in chosen_products:
            quantity = random.randint(1, 5)
            if random.random() < 0.03:
                quantity = -abs(quantity)

            cost = product_cost[product_id]
            unit_price = round(cost * random.uniform(1.2, 2.5), 2)  
            discount_percent = random.choices(
                [0, random.randint(1, 15), random.randint(16, 40), random.randint(41, 100)],
                weights=[0.5, 0.3, 0.15, 0.05],
            )[0]

            rows.append(
                {
                    "item_id": item_id,
                    "order_id": order["order_id"],
                    "product_id": product_id,
                    "quantity": quantity,
                    "unit_price": unit_price,
                    "discount_percent": discount_percent,
                }
            )
            item_id += 1


    n_orphans = max(1, int(len(rows) * 0.01))
    for _ in range(n_orphans):
        fake_order_id = max_valid_order_id + random.randint(1000, 9000)
        product_id = random.choice(product_ids)
        rows.append(
            {
                "item_id": item_id,
                "order_id": fake_order_id,
                "product_id": product_id,
                "quantity": random.randint(1, 3),
                "unit_price": round(product_cost[product_id] * 1.5, 2),
                "discount_percent": 0,
            }
        )
        item_id += 1

    random.shuffle(rows) 

    with open(f"{OUT_DIR}/order_items.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)

    print(f"order_items.csv: {len(rows)} rows written ({n_orphans} intentional orphan rows)")
    return rows


def main():
    import os

    os.makedirs(OUT_DIR, exist_ok=True)
    customers = generate_customers()
    products = generate_products()
    orders = generate_orders(customers)
    generate_order_items(orders, products)
    print("\nData generation complete. Files written to:", OUT_DIR)


if __name__ == "__main__":
    main()

customers.csv: 550 rows written
products.csv: 520 rows written
orders.csv: 900 rows written
order_items.csv: 2240 rows written (22 intentional orphan rows)

Data generation complete. Files written to: /Workspace/Users/yashikumawat53@gmail.com/Drafts/ecommerce_project/data


In [0]:
from pyspark.sql import functions as F
customers = spark.read.option("header", True).csv("/Workspace/Users/yashikumawat53@gmail.com/Drafts/ecommerce_project/data/customers.csv")
products = spark.read.option("header", True).csv("/Workspace/Users/yashikumawat53@gmail.com/Drafts/ecommerce_project/data/products.csv")
orders = spark.read.option("header", True).csv("/Workspace/Users/yashikumawat53@gmail.com/Drafts/ecommerce_project/data/orders.csv")
order_items = spark.read.option("header", True).csv("/Workspace/Users/yashikumawat53@gmail.com/Drafts/ecommerce_project/data/order_items.csv")

NULL_LIKE_STRINGS = ["", "NULL", "null", "None", "NA", "N/A", "nan", "NaN"]

def check_nulls(df, name):
    print(f"\n{name}")

    result = df.select([
        F.count(
            F.when(
                F.col(c).isNull() |
                F.trim(F.col(c)).isin(NULL_LIKE_STRINGS),
                c
            )
        ).alias(c)
        for c in df.columns
    ])

    result.show(truncate=False)

# Check all datasets
check_nulls(customers, "Customers")
check_nulls(products, "Products")
check_nulls(orders, "Orders")
check_nulls(order_items, "Order Items")


Customers
+-----------+-------------+-----+-----------------+-------------+
|customer_id|customer_name|email|registration_date|customer_type|
+-----------+-------------+-----+-----------------+-------------+
|0          |0            |0    |0                |0            |
+-----------+-------------+-----+-----------------+-------------+


Products
+----------+------------+--------+-----------+----------+
|product_id|product_name|category|subcategory|cost_price|
+----------+------------+--------+-----------+----------+
|0         |0           |0       |0          |0         |
+----------+------------+--------+-----------+----------+


Orders
+--------+-----------+----------+------+-----------+
|order_id|customer_id|order_date|status|region_code|
+--------+-----------+----------+------+-----------+
|0       |48         |0         |0     |0          |
+--------+-----------+----------+------+-----------+


Order Items
+-------+--------+----------+--------+----------+----------------+
|it